# Dimension 3.1 - Health Resource Allocation Gap Analysis
Needs-based formula, gap grading (A-E), diverging choropleth.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np

from hdi.config import *
from hdi.data.loaders import load_health_expenditure, load_daly, load_demographics
from hdi.visualization.maps import choropleth_diverging

## 1. Needs-Based Formula

In [ ]:
# Load data
health_exp = load_health_expenditure()
daly = load_daly()
demographics = load_demographics()

# Global total health expenditure
global_health_exp = health_exp["health_exp_usd"].sum()
global_daly = daly["daly_total"].sum()

# Theoretical need per country:
# TheoreticalNeed_i = (DALY_i / DALY_global) * GlobalHealthExp
# Weighted by NCD share and age structure
daly["daly_share"] = daly["daly_total"] / global_daly

# NCD weight: countries with higher NCD burden need more per-DALY spending
ncd_weight = daly["ncd_share"].fillna(0.5)
age_weight = demographics["aged_dependency_ratio"].fillna(1.0)

# Composite weight (normalize so weights sum to N)
composite_weight = ncd_weight * age_weight
composite_weight = composite_weight / composite_weight.mean()

# Theoretical need
theoretical_need = daly["daly_share"] * global_health_exp * composite_weight
theoretical_need.name = "theoretical_need_usd"

print(f"Global health expenditure: ${global_health_exp:,.0f}")
print(f"Global DALYs: {global_daly:,.0f}")
theoretical_need.describe()

## 2. Gap Calculation & Grading

In [ ]:
# Gap = Actual expenditure - Theoretical need
# Positive gap = over-resourced, Negative gap = under-resourced
gap = health_exp["health_exp_usd"] - theoretical_need
gap.name = "resource_gap_usd"

# Quintile-based grading: A (most over-resourced) through E (most under-resourced)
grade_labels = ["E", "D", "C", "B", "A"]
gap_grade = pd.qcut(gap, q=5, labels=grade_labels)
gap_grade.name = "gap_grade"

# Combine into a summary DataFrame
gap_df = pd.DataFrame({
    "actual_exp": health_exp["health_exp_usd"],
    "theoretical_need": theoretical_need,
    "gap": gap,
    "grade": gap_grade
})

print("Grade distribution:")
print(gap_df["grade"].value_counts().sort_index())
print()
gap_df.sort_values("gap").head(10)

## 3. Diverging Choropleth Map

In [ ]:
# Plot diverging choropleth: red = under-resourced (E), blue = over-resourced (A)
fig = choropleth_diverging(
    data=gap_df,
    value_col="gap",
    title="Health Resource Allocation Gap (Actual - Needs-Based Theoretical)",
    colorbar_label="Gap (USD)",
    cmap="RdBu",
    center=0
)
fig